# Training

### Libraries and constants

In [3]:
import pandas as pd
from pathlib import Path
from joblib import parallel_backend
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

In [11]:
# --- Settings ---
pd.set_option('display.float_format', lambda x: f'{x:.6f}') # turn off exponent format (1e-12) for data frames, use float format instead

In [9]:
# --- CONSTANTS ---
RANDOM_SEED = 42
TARGET_NAME = 'state'

## Data import

In [7]:
df_train = pd.read_csv(Path('data') / '3 df_train_post_EDA.csv')
df_test = pd.read_csv(Path('data') / '3 df_test_post_EDA.csv')

In [10]:
X_train, y_train = df_train.drop(columns=[TARGET_NAME]), df_train[TARGET_NAME]
X_test, y_test = df_test.drop(columns=[TARGET_NAME]), df_test[TARGET_NAME]

## Training

### SVM

The Tuning Philosophy: SVMs are mathematically rigid. Their performance hinges heavily on a small set of interacting continuous parameters—primarily the regularization parameter ($C$) and the kernel coefficient ($\gamma$). Because the parameter space is relatively small but highly sensitive to specific combinations, we use GridSearchCV to exhaustively test every specified combination.

In [ ]:
# 1. Define your pipeline and grid
svm_pipeline = Pipeline([
    ('svm', SVC(class_weight='balanced', random_state=RANDOM_SEED)),
])

svm_param_grid = [
    {'svm__kernel': ['linear'], 'svm__C': [0.1, 1, 2, 3]},
    {'svm__kernel': ['rbf'], 'svm__C': [0.1, 1, 2, 3], 'svm__gamma': ['scale', 'auto', 0.1, 0.5, 1]},
    {'svm__kernel': ['poly'], 'svm__C': [0.1, 1, 2, 3], 'svm__degree': [2, 3]},
]

tscv = TimeSeriesSplit(n_splits=5)

# 2. Setup grid search with verbose=3
svm_grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=3,
)

# 3. Fit the model using threading backend so prints show up live
print("Starting grid search...")
with parallel_backend('threading'):
    svm_grid_search.fit(X_train, y_train)

# 4. Print the best training results
print(f"\nBest estimator: {svm_grid_search.best_estimator_}\n")
print(f"Best SVM Parameters: {svm_grid_search.best_params_}")
print(f"Best SVM CV Score: {svm_grid_search.best_score_:.4f}\n")

# 5. Predict on your test dataset
print("Evaluating on test data...")
y_pred = svm_grid_search.predict(X_test)

print("\n=== TEST SET CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

### Trees

The Tuning Philosophy: Random Forests have a massive hyperparameter space (n_estimators, max_depth, min_samples_split, max_features, etc.). Running an exhaustive grid search on a forest is computationally crippling and mathematically unnecessary. Research shows that testing a random sample of parameter combinations (RandomizedSearchCV) yields near-optimal results in a fraction of the time, because only a few hyperparameters truly dictate the model's success.

In [ ]:
rf_pipeline = Pipeline([
    ('rf', RandomForestClassifier(class_weight="balanced", random_state=RANDOM_SEED)),
])

rf_param_dist = {
    'rf__n_estimators': np.arange(100, 1001, 100),
    'rf__max_depth': [None, 10, 20, 30, 40, 50],
    'rf__min_samples_split': np.arange(2, 21, 2),
    'rf__min_samples_leaf': np.arange(1, 11, 2),
    'rf__max_features': ['sqrt', 'log2', None],
}

rf_random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=30,          # 30 random combinations to run (out of 10 * 6 * 10 * 5 * 3 = 2,700 options)
    cv=TimeSeriesSplit(n_splits=5),
    scoring='f1_macro',
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=3,
)

print("Starting randomized grid search...")
with parallel_backend('threading'):
    rf_random_search.fit(X_train, y_train)

print(f"Best estimator: {rf_random_search.best_estimator_}\n")
print(f"Best RF Parameters: {rf_random_search.best_params_}")
print(f"Best RF CV Score: {rf_random_search.best_score_:.4f}\n")

print("Evaluating on test data...")
y_pred = rf_random_search.predict(X_test)

print("\n=== TEST SET CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

Starting randomized grid search...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV 1/5] END rf__max_depth=None, rf__max_features=log2, rf__min_samples_leaf=7, rf__min_samples_split=14, rf__n_estimators=100;, score=0.946 total time=   1.8s
[CV 1/5] END rf__max_depth=30, rf__max_features=log2, rf__min_samples_leaf=7, rf__min_samples_split=20, rf__n_estimators=100;, score=0.946 total time=   1.8s
[CV 2/5] END rf__max_depth=None, rf__max_features=log2, rf__min_samples_leaf=7, rf__min_samples_split=14, rf__n_estimators=100;, score=0.977 total time=   2.2s
[CV 2/5] END rf__max_depth=30, rf__max_features=log2, rf__min_samples_leaf=7, rf__min_samples_split=20, rf__n_estimators=100;, score=0.969 total time=   2.3s
[CV 1/5] END rf__max_depth=40, rf__max_features=None, rf__min_samples_leaf=5, rf__min_samples_split=16, rf__n_estimators=100;, score=0.915 total time=   2.5s
[CV 3/5] END rf__max_depth=None, rf__max_features=log2, rf__min_samples_leaf=7, rf__min_samples_split=14, rf_